# 🤖 AI-Powered Paraphrasing Tool

**Author:** Mohammed Hasif
**Date:** April 2026  
**Model:** T5 Transformer (Vamsi/T5_Paraphrase_Paws)

---

## 📌 Project Overview

This notebook builds a complete AI-powered paraphrasing tool using:
- **T5 Transformer** (HuggingFace) for paraphrase generation
- **LanguageTool + TextBlob** for grammar and spelling checks
- **spaCy** for fluency scoring
- **SBERT** for semantic similarity evaluation
- **BLEU + ROUGE** for originality measurement

### Pipeline Flow
Input Text → Generate Paraphrases → Quality Check → Evaluate Metrics → Rank & Return Best

---

## ⚙️ Part 1: Install Dependencies

Install all required libraries. **Restart the runtime after this cell completes.**

In [14]:
# ============================================================
# PART 1: Install all required libraries
# Run this cell first and restart runtime if prompted
# ============================================================

!pip install transformers torch sentencepiece
!pip install language-tool-python          # Grammar checker
!pip install textblob                      # Spelling + basic NLP
!pip install rouge-score                   # ROUGE evaluation metric
!pip install sentence-transformers         # Semantic similarity (SBERT)
!pip install nltk                          # BLEU score + tokenization
!pip install spacy
!python -m spacy download en_core_web_sm   # spaCy English model

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

print("✅ All dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All dependencies installed successfully!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 🔧 Part 2: Paraphrasing Engine

**Model used:** `Vamsi/T5_Paraphrase_Paws`  
A T5 model fine-tuned on the PAWS (Paraphrase Adversaries from Word Scrambling) dataset.

### How it works
- Input text is prefixed with `"paraphrase: ... </s>"` as T5 expects
- **Nucleus sampling (top-p = 0.92)** generates diverse outputs
- A **temperature ladder** (1.2 → 1.6 → 2.0) produces variants ranging from conservative to creative
- Deduplication removes near-identical results

### ⚠️ Note
First run downloads ~900MB model weights — cached for all future runs.

In [6]:
# PART 2 (CLEAN): Paraphrasing Engine
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

class ParaphrasingEngine:

    def __init__(self, model_name="Vamsi/T5_Paraphrase_Paws"):
        print(f"Loading model: {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        print(f"Model loaded on {self.device.upper()}")

    def _encode(self, text):
        return self.tokenizer(
            f"paraphrase: {text} </s>",
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding="max_length"
        ).to(self.device)

    def _deduplicate(self, candidates, original):
        seen, unique = set(), []
        orig_norm = original.lower().strip(" .")
        for c in candidates:
            norm = c.lower().strip(" .")
            if norm not in seen and norm != orig_norm:
                seen.add(norm)
                unique.append(c)
        return unique

    def paraphrase(self, text, num_variants=3, max_length=256):
        enc = self._encode(text)
        temperatures = [1.2 + i * 0.4 for i in range(num_variants + 2)]
        results = []

        for idx, temp in enumerate(temperatures):
            torch.manual_seed(42 + idx * 7)
            output = self.model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_length=max_length,
                do_sample=True,
                top_p=0.92,
                top_k=120,
                temperature=temp,
                no_repeat_ngram_size=3,
                num_return_sequences=1,
                early_stopping=True,
            )
            decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)
            results.append(decoded)

        unique = self._deduplicate(results, text)
        return unique[:num_variants] if len(unique) >= num_variants else unique


# Load model
engine = ParaphrasingEngine()

# Test
test_sentences = [
    "The quick brown fox jumps over the lazy dog near the riverbank.",
    "Artificial intelligence is rapidly changing the way we work and live.",
    "She decided to leave the company after ten years of dedicated service."
]

print("\n" + "="*60)
for sentence in test_sentences:
    variants = engine.paraphrase(sentence, num_variants=3)
    print(f"\nOriginal:\n  {sentence}")
    print(f"\nParaphrases:")
    for i, v in enumerate(variants, 1):
        print(f"  [{i}] {v}")
    print("-"*60)

print("\nPart 2 complete!")

Loading model: Vamsi/T5_Paraphrase_Paws ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded on CPU


Original:
  The quick brown fox jumps over the lazy dog near the riverbank.

Paraphrases:
  [1] The brown flash jumps over the lazy dog in the riverbank .
  [2] During the rush, the swift brown fox hops nearby onto the lazy dog.
  [3] The quick brown fox skips over atfuriated dog's penneck sped near what will not happen further down .
------------------------------------------------------------

Original:
  Artificial intelligence is rapidly changing the way we work and live.

Paraphrases:
  [1] The Artificial Intelligence revolution is rapidly changing the way we live and work.
  [2] ? Machine learning is drastically changing the way we work and live ...
  [3] It depends ever more on artificial intelligence to change how you grow up and operate daily.
------------------------------------------------------------

Original:
  She decided to leave the company after ten years of dedicated service.

Paraphrases:
  [1] After ten years of dedicated service she decided t

## ✅ Part 3: Quality Checker

Three independent checks run on every paraphrase variant:

| Check | Tool | What it measures |
|-------|------|-----------------|
| Grammar | LanguageTool | Grammatical errors and suggested corrections |
| Spelling | TextBlob | Misspelled words with auto-correction |
| Fluency | spaCy | Sentence structure, verb presence, dependency depth |

A variant is marked **PASS** only if:
- Grammar errors = 0
- Spelling errors = 0  
- Fluency score > 0.60

The grammar corrector automatically fixes minor issues in the final output.

In [7]:
# PART 3: Quality Checker — Grammar, Spelling, Fluency
import language_tool_python
from textblob import TextBlob
import spacy

class QualityChecker:

    def __init__(self):
        print("Loading quality checker tools...")
        self.grammar_tool = language_tool_python.LanguageTool('en-US')
        self.nlp = spacy.load("en_core_web_sm")
        print("Quality checker ready!")

    def check_grammar(self, text):
        matches = self.grammar_tool.check(text)
        issues = [
            {
                "message": m.message,
                "suggestion": m.replacements[:3] if m.replacements else [],
                "context": m.context
            }
            for m in matches
        ]
        corrected = language_tool_python.utils.correct(text, matches)
        return {
            "error_count": len(issues),
            "issues": issues,
            "corrected": corrected
        }

    def check_spelling(self, text):
        blob = TextBlob(text)
        corrected = str(blob.correct())
        orig_words = text.split()
        corr_words = corrected.split()
        # Only compare up to the shorter length to avoid index errors
        min_len = min(len(orig_words), len(corr_words))
        misspelled = [
            (orig_words[i], corr_words[i])
            for i in range(min_len)
            if orig_words[i] != corr_words[i]
        ]
        return {
            "misspelled_count": len(misspelled),
            "corrections": misspelled,
            "corrected_text": corrected
        }

    def check_fluency(self, text):
        doc = self.nlp(text)
        sentences = list(doc.sents)
        if not sentences:
            return {"fluency_score": 0.0, "interpretation": "Poor", "notes": ["No sentences detected."]}

        scores, notes = [], []
        for sent in sentences:
            tokens = [t for t in sent if not t.is_punct]
            if len(tokens) < 4:
                scores.append(0.4)
                notes.append(f"Too short: '{sent.text.strip()}'")
                continue
            has_verb = any(t.pos_ == "VERB" for t in sent)
            if not has_verb:
                scores.append(0.5)
                notes.append(f"No verb: '{sent.text.strip()}'")
                continue
            depths = [len(list(t.ancestors)) for t in sent]
            avg_depth = sum(depths) / len(depths) if depths else 0
            depth_score = min(avg_depth / 5.0, 1.0)
            scores.append(0.5 + 0.5 * depth_score)

        fluency = round(sum(scores) / len(scores), 3)
        interp = ("Excellent" if fluency > 0.8 else
                  "Good"      if fluency > 0.6 else
                  "Fair"      if fluency > 0.4 else "Poor")
        return {"fluency_score": fluency, "interpretation": interp, "notes": notes}

    def full_report(self, text):
        grammar  = self.check_grammar(text)
        spelling = self.check_spelling(text)
        fluency  = self.check_fluency(text)
        overall  = (grammar["error_count"] == 0
                    and spelling["misspelled_count"] == 0
                    and fluency["fluency_score"] > 0.6)
        return {
            "input_text":  text,
            "grammar":     grammar,
            "spelling":    spelling,
            "fluency":     fluency,
            "overall_clean": overall
        }


# Load checker
checker = QualityChecker()

# Test on ALL variants from Part 2
test_inputs = [
    ("The quick brown fox jumps over the lazy dog near the riverbank.",
     ["The brown flash jumps over the lazy dog in the riverbank .",
      "During the rush, the swift brown fox hops nearby onto the lazy dog.",
      "The quick brown fox skips over atfuriated dog's penneck sped near what will not happen further down ."]),

    ("Artificial intelligence is rapidly changing the way we work and live.",
     ["The Artificial Intelligence revolution is rapidly changing the way we live and work.",
      "? Machine learning is drastically changing the way we work and live ...",
      "It depends ever more on artificial intelligence to change how you grow up and operate daily."]),

    ("She decided to leave the company after ten years of dedicated service.",
     ["After ten years of dedicated service she decided to leave the company .",
      "10) after 10 years spent doing dedicated duty determined to quit the firm",
      "It has decision of leaving for ten further distinguished companies today."])
]

print("\n" + "="*62)
for original, variants in test_inputs:
    print(f"\nOriginal: {original}")
    print("-"*62)
    for i, para in enumerate(variants, 1):
        report = checker.full_report(para)
        g = report["grammar"]
        s = report["spelling"]
        f = report["fluency"]
        flag = "PASS" if report["overall_clean"] else "REVIEW"
        print(f"\n  Variant [{i}]: {para}")
        print(f"    Grammar errors   : {g['error_count']}")
        print(f"    Spelling errors  : {s['misspelled_count']}")
        print(f"    Fluency score    : {f['fluency_score']} ({f['interpretation']})")
        print(f"    Corrected text   : {g['corrected']}")
        print(f"    Status           : {'✅ ' + flag if flag == 'PASS' else '⚠️  ' + flag}")
    print("="*62)

print("\nPart 3 complete!")

Loading quality checker tools...


Quality checker ready!


Original: The quick brown fox jumps over the lazy dog near the riverbank.
--------------------------------------------------------------

  Variant [1]: The brown flash jumps over the lazy dog in the riverbank .
    Grammar errors   : 1
    Spelling errors  : 0
    Fluency score    : 0.725 (Good)
    Corrected text   : The brown flash jumps over the lazy dog in the riverbank.
    Status           : ⚠️  REVIEW

  Variant [2]: During the rush, the swift brown fox hops nearby onto the lazy dog.
    Grammar errors   : 0
    Spelling errors  : 1
    Fluency score    : 0.667 (Good)
    Corrected text   : During the rush, the swift brown fox hops nearby onto the lazy dog.
    Status           : ⚠️  REVIEW

  Variant [3]: The quick brown fox skips over atfuriated dog's penneck sped near what will not happen further down .
    Grammar errors   : 3
    Spelling errors  : 3
    Fluency score    : 0.753 (Good)
    Corrected text   : The quick brown fox skips over infuriate

## 📊 Part 4: Evaluation Metrics

Three metrics evaluate paraphrase quality from different angles:

### BLEU Score (Bilingual Evaluation Understudy)
Measures n-gram overlap between original and paraphrase.
- **Ideal range: 0.10 – 0.50**
- Too high (> 0.75) = near-copy, not a true paraphrase
- Too low (< 0.10) = meaning may be lost

### ROUGE Score (Recall-Oriented Understudy for Gisting Evaluation)
Measures recall-based overlap using unigrams, bigrams, and longest common subsequence.
- ROUGE-1: unigram overlap
- ROUGE-2: bigram overlap
- ROUGE-L: longest common subsequence

### Semantic Similarity (SBERT)
Uses `all-MiniLM-L6-v2` sentence embeddings to measure cosine similarity.
- **Ideal: > 0.70** — confirms meaning is preserved despite different wording

In [8]:
# PART 4: Evaluation Metrics — BLEU, ROUGE, Semantic Similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
import nltk

class EvaluationMetrics:

    def __init__(self):
        print("Loading SBERT model for semantic similarity...")
        self.sbert    = SentenceTransformer('all-MiniLM-L6-v2')
        self.rouge    = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.smoother = SmoothingFunction().method1
        print("Evaluation module ready!")

    def bleu_score(self, original, paraphrase):
        ref   = [nltk.word_tokenize(original.lower())]
        hyp   =  nltk.word_tokenize(paraphrase.lower())
        score = sentence_bleu(ref, hyp, smoothing_function=self.smoother)
        return round(score, 4)

    def rouge_scores(self, original, paraphrase):
        scores = self.rouge.score(original, paraphrase)
        return {
            "rouge1": round(scores["rouge1"].fmeasure, 4),
            "rouge2": round(scores["rouge2"].fmeasure, 4),
            "rougeL": round(scores["rougeL"].fmeasure, 4),
        }

    def semantic_similarity(self, original, paraphrase):
        e1    = self.sbert.encode(original,   convert_to_tensor=True)
        e2    = self.sbert.encode(paraphrase, convert_to_tensor=True)
        score = util.cos_sim(e1, e2).item()
        return round(score, 4)

    def originality_label(self, bleu):
        if bleu < 0.20:   return "High originality"
        elif bleu < 0.50: return "Good balance"
        elif bleu < 0.75: return "Moderate"
        else:             return "Low originality (near-copy)"

    def quality_flag(self, bleu, sem_sim):
        if sem_sim > 0.70 and 0.10 < bleu < 0.75:
            return "GOOD"
        elif sem_sim < 0.50:
            return "MEANING DRIFT"
        elif bleu >= 0.75:
            return "NEAR-COPY"
        else:
            return "REVIEW"

    def evaluate(self, original, paraphrase):
        bleu  = self.bleu_score(original, paraphrase)
        rouge = self.rouge_scores(original, paraphrase)
        sem   = self.semantic_similarity(original, paraphrase)
        flag  = self.quality_flag(bleu, sem)
        return {
            "bleu":        bleu,
            "rouge":       rouge,
            "semantic":    sem,
            "originality": self.originality_label(bleu),
            "flag":        flag
        }


# Load evaluator
evaluator = EvaluationMetrics()

# Use the same variants from Part 2
test_data = [
    ("The quick brown fox jumps over the lazy dog near the riverbank.",
     ["The brown flash jumps over the lazy dog in the riverbank .",
      "During the rush, the swift brown fox hops nearby onto the lazy dog.",
      "The quick brown fox skips over atfuriated dog's penneck sped near what will not happen further down ."]),

    ("Artificial intelligence is rapidly changing the way we work and live.",
     ["The Artificial Intelligence revolution is rapidly changing the way we live and work.",
      "? Machine learning is drastically changing the way we work and live ...",
      "It depends ever more on artificial intelligence to change how you grow up and operate daily."]),

    ("She decided to leave the company after ten years of dedicated service.",
     ["After ten years of dedicated service she decided to leave the company .",
      "10) after 10 years spent doing dedicated duty determined to quit the firm",
      "It has decision of leaving for ten further distinguished companies today."])
]

print("\n" + "="*65)
print(f"  {'Var':<5} {'BLEU':<8} {'R-1':<8} {'R-L':<8} {'Sem.Sim':<10} {'Flag'}")
print("="*65)

for original, variants in test_data:
    print(f"\nOriginal: {original}")
    print("-"*65)
    for i, para in enumerate(variants, 1):
        ev = evaluator.evaluate(original, para)
        r  = ev["rouge"]
        icon = "✅" if ev["flag"] == "GOOD" else "⚠️ "
        print(f"  [{i}]  {ev['bleu']:<8} {r['rouge1']:<8} {r['rougeL']:<8} "
              f"{ev['semantic']:<10} {icon} {ev['flag']}")
        print(f"        {para[:70]}{'...' if len(para)>70 else ''}")
        print(f"        Originality: {ev['originality']}")
    print()

print("="*65)
print("\nScore guide:")
print("  BLEU      : 0.1-0.5 ideal  | >0.75 = near-copy | <0.1 = meaning lost")
print("  Sem.Sim   : >0.70 ideal    | <0.50 = meaning drift")
print("  ROUGE-L   : measures longest common subsequence overlap")
print("\nPart 4 complete!")

Loading SBERT model for semantic similarity...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Evaluation module ready!

  Var   BLEU     R-1      R-L      Sem.Sim    Flag

Original: The quick brown fox jumps over the lazy dog near the riverbank.
-----------------------------------------------------------------
  [1]  0.4125   0.7826   0.7826   0.8106     ✅ GOOD
        The brown flash jumps over the lazy dog in the riverbank .
        Originality: Good balance
  [2]  0.0925   0.56     0.48     0.7554     ⚠️  REVIEW
        During the rush, the swift brown fox hops nearby onto the lazy dog.
        Originality: High originality
  [3]  0.1507   0.4667   0.4667   0.6259     ⚠️  REVIEW
        The quick brown fox skips over atfuriated dog's penneck sped near what...
        Originality: High originality


Original: Artificial intelligence is rapidly changing the way we work and live.
-----------------------------------------------------------------
  [1]  0.4355   0.9167   0.75     0.94       ✅ GOOD
        The Artificial Intelligence revolution is rapidly changing the way we ...
 

## 🚀 Part 5: Full Pipeline

Combines all modules into one end-to-end tool:

**Input Text**
↓
**[ParaphrasingEngine]** → generates N variants
↓
**[QualityChecker]** → grammar + spelling + fluency per variant
↓
**[EvaluationMetrics]** → BLEU + ROUGE + Semantic Similarity per variant
↓
**[Composite Scorer]** → 50% Semantic + 30% Fluency − Grammar Penalty
↓
**Ranked Output** → Best variant auto-selected

> ⚠️ **Note: No modules are reloaded** — this cell reuses `engine`, `checker`, and `evaluator` from Parts 2–4.

In [9]:
# PART 5: Full Pipeline — Generate → Check → Evaluate → Rank
class ParaphrasingTool:

    def __init__(self, engine, checker, evaluator):
        self.engine    = engine
        self.checker   = checker
        self.evaluator = evaluator
        print("Full pipeline ready!")

    def run(self, text, num_variants=3):
        print("\n" + "="*65)
        print(f"INPUT: {text}")
        print("="*65)

        # Step 1: Generate
        print("\nGenerating paraphrases...")
        paraphrases = self.engine.paraphrase(text, num_variants=num_variants)

        candidates = []
        for i, para in enumerate(paraphrases, 1):

            # Step 2: Quality check
            quality = self.checker.full_report(para)

            # Step 3: Evaluate metrics
            metrics = self.evaluator.evaluate(text, para)

            # Step 4: Composite score
            # 50% semantic similarity (meaning preserved)
            # 30% fluency
            # 20% penalty for grammar errors (capped at 0.20)
            grammar_penalty = min(quality["grammar"]["error_count"] * 0.05, 0.20)
            composite = round(
                0.50 * metrics["semantic"]
              + 0.30 * quality["fluency"]["fluency_score"]
              - grammar_penalty, 3
            )

            # Step 5: Use grammar-corrected text as final output
            final_text = quality["grammar"]["corrected"]

            candidates.append({
                "variant":    i,
                "original_para": para,
                "final_text": final_text,
                "composite":  composite,
                "semantic":   metrics["semantic"],
                "bleu":       metrics["bleu"],
                "rougeL":     metrics["rouge"]["rougeL"],
                "fluency":    quality["fluency"]["fluency_score"],
                "fluency_label": quality["fluency"]["interpretation"],
                "grammar_errors":  quality["grammar"]["error_count"],
                "spelling_errors": quality["spelling"]["misspelled_count"],
                "metric_flag": metrics["flag"],
                "overall_clean": quality["overall_clean"]
            })

        # Step 6: Rank by composite score
        candidates.sort(key=lambda x: x["composite"], reverse=True)
        for rank, c in enumerate(candidates, 1):
            c["rank"] = rank

        # Print ranked results
        print(f"\n{'Rank':<6} {'Score':<8} {'Sem':<8} {'BLEU':<8} {'Fluency':<10} {'G.Err':<7} {'Flag'}")
        print("-"*65)
        for c in candidates:
            crown = " 👑" if c["rank"] == 1 else ""
            print(f"  [{c['rank']}]  {c['composite']:<8} {c['semantic']:<8} "
                  f"{c['bleu']:<8} {c['fluency_label']:<10} "
                  f"{c['grammar_errors']:<7} {c['metric_flag']}{crown}")
            print(f"       {c['final_text'][:70]}{'...' if len(c['final_text'])>70 else ''}")

        best = candidates[0]
        print(f"\n{'='*65}")
        print(f"BEST PARAPHRASE:")
        print(f"  {best['final_text']}")
        print(f"  Composite: {best['composite']}  |  "
              f"Semantic: {best['semantic']}  |  "
              f"Fluency: {best['fluency_label']}")
        print(f"{'='*65}\n")

        return candidates


# Build pipeline from already-loaded modules (no reloading!)
tool = ParaphrasingTool(engine, checker, evaluator)

# Run on 4 varied sentences
sentences = [
    "The quick brown fox jumps over the lazy dog near the riverbank.",
    "Artificial intelligence is rapidly changing the way we work and live.",
    "She decided to leave the company after ten years of dedicated service.",
    "Climate change poses a serious threat to biodiversity and human health worldwide."
]

all_results = {}
for s in sentences:
    all_results[s] = tool.run(s, num_variants=3)

print("\nPart 5 complete!")

Full pipeline ready!

INPUT: The quick brown fox jumps over the lazy dog near the riverbank.

Generating paraphrases...

Rank   Score    Sem      BLEU     Fluency    G.Err   Flag
-----------------------------------------------------------------
  [1]  0.578    0.7554   0.0925   Good       0       REVIEW 👑
       During the rush, the swift brown fox hops nearby onto the lazy dog.
  [2]  0.573    0.8106   0.4125   Good       1       GOOD
       The brown flash jumps over the lazy dog in the riverbank.
  [3]  0.389    0.6259   0.1507   Good       3       REVIEW
       The quick brown fox skips over infuriated dog's pen neck sped near wha...

BEST PARAPHRASE:
  During the rush, the swift brown fox hops nearby onto the lazy dog.
  Composite: 0.578  |  Semantic: 0.7554  |  Fluency: Good


INPUT: Artificial intelligence is rapidly changing the way we work and live.

Generating paraphrases...

Rank   Score    Sem      BLEU     Fluency    G.Err   Flag
-------------------------------------------

## 📈 Part 6: Final Evaluation Report

Aggregates results across all test sentences and produces a summary report.

### Scoring Targets
| Metric | Target | Meaning |
|--------|--------|---------|
| Semantic Similarity | > 0.70 | Meaning preserved |
| BLEU Score | 0.10 – 0.50 | Genuine paraphrase |
| Fluency Score | > 0.60 | Natural, readable output |
| Composite Score | > 0.60 | Overall quality |

### Status Labels
- ✅ **GOOD** — Passes all targets
- ⚠️ **NEAR-COPY** — BLEU > 0.75, too similar to original
- ⚠️ **REVIEW** — One or more targets missed
- ⚠️ **MEANING DRIFT** — Semantic similarity < 0.50

In [10]:
# PART 6: Summary Evaluation Report
print("\n" + "="*65)
print("        FINAL EVALUATION REPORT — AI Paraphrasing Tool")
print("="*65)

total, passed, near_copy, drift = 0, 0, 0, 0

for original, candidates in all_results.items():
    best = candidates[0]   # Already ranked — [0] is best
    total += 1

    if best["metric_flag"] == "GOOD":        passed    += 1
    elif best["metric_flag"] == "NEAR-COPY": near_copy += 1
    elif best["metric_flag"] == "MEANING DRIFT": drift += 1

    print(f"\nInput   : {original[:65]}{'...' if len(original)>65 else ''}")
    print(f"Best    : {best['final_text'][:65]}{'...' if len(best['final_text'])>65 else ''}")
    print(f"Scores  : Composite={best['composite']}  "
          f"Semantic={best['semantic']}  "
          f"BLEU={best['bleu']}  "
          f"ROUGE-L={best['rougeL']}")
    print(f"Quality : Fluency={best['fluency_label']}  "
          f"Grammar errors={best['grammar_errors']}  "
          f"Spelling errors={best['spelling_errors']}")
    status = ("✅ GOOD" if best["metric_flag"] == "GOOD"
              else "⚠️  " + best["metric_flag"])
    print(f"Status  : {status}")

# Aggregate stats
print("\n" + "="*65)
print("  AGGREGATE STATISTICS")
print("-"*65)
print(f"  Total sentences tested  : {total}")
print(f"  Best variants — GOOD    : {passed}  ({round(passed/total*100)}%)")
print(f"  Best variants — NEAR-COPY: {near_copy} ({round(near_copy/total*100)}%)")
print(f"  Best variants — DRIFT   : {drift}  ({round(drift/total*100)}%)")

# Score averages across best variants
avg_sem  = round(sum(r[0]["semantic"]   for r in all_results.values()) / total, 3)
avg_bleu = round(sum(r[0]["bleu"]       for r in all_results.values()) / total, 3)
avg_flu  = round(sum(r[0]["fluency"]    for r in all_results.values()) / total, 3)
avg_comp = round(sum(r[0]["composite"]  for r in all_results.values()) / total, 3)

print(f"\n  Average scores (best variants only):")
print(f"    Semantic Similarity : {avg_sem}   (target > 0.70)")
print(f"    BLEU Score          : {avg_bleu}  (target 0.10–0.50)")
print(f"    Fluency Score       : {avg_flu}   (target > 0.60)")
print(f"    Composite Score     : {avg_comp}")

print("\n  Interpretation guide:")
print("    Semantic > 0.70  → meaning well preserved")
print("    BLEU 0.10-0.50   → genuine paraphrase, not a copy")
print("    Fluency > 0.60   → grammatically natural output")
print("="*65)
print("\nAll parts complete! Your AI Paraphrasing Tool is fully built.")


        FINAL EVALUATION REPORT — AI Paraphrasing Tool

Input   : The quick brown fox jumps over the lazy dog near the riverbank.
Best    : During the rush, the swift brown fox hops nearby onto the lazy do...
Scores  : Composite=0.578  Semantic=0.7554  BLEU=0.0925  ROUGE-L=0.48
Quality : Fluency=Good  Grammar errors=0  Spelling errors=1
Status  : ⚠️  REVIEW

Input   : Artificial intelligence is rapidly changing the way we work and l...
Best    : The Artificial Intelligence revolution is rapidly changing the wa...
Scores  : Composite=0.674  Semantic=0.94  BLEU=0.4355  ROUGE-L=0.75
Quality : Fluency=Good  Grammar errors=0  Spelling errors=0
Status  : ✅ GOOD

Input   : She decided to leave the company after ten years of dedicated ser...
Best    : After ten years of dedicated service she decided to leave the com...
Scores  : Composite=0.66  Semantic=0.9917  BLEU=0.7765  ROUGE-L=0.5
Quality : Fluency=Good  Grammar errors=1  Spelling errors=0
Status  : ⚠️  NEAR-COPY

Input   : Climate chang

In [ ]:
# This saves the current running notebook to /content/
from google.colab import _message
import json

# Request the current notebook content from Colab
notebook_json = _message.blocking_request(
    "get_ipynb", request="", timeout_sec=120
)

# Save it to /content/
save_path = "/content/AI_Paraphrasing_Tool.ipynb"
with open(save_path, "w") as f:
    json.dump(notebook_json["ipynb"], f, indent=1)

print(f"✅ Notebook saved to: {save_path}")